# Capacity allocation with dependent demands: buy-up

The previous notebook assumed the two fare classes were independent — closing the discount fare had no effect on full-fare demand. That is rarely true. When discount seats run out, some turned-away shoppers don't simply disappear: a fraction **buy up** to the full fare instead.

We model this with a recapture rate $\alpha = 0.3$: of the discount demand that cannot be accommodated, $30\%$ spills over into full-fare demand. Everything else matches the independent case (capacity $C = 100$, fares $p_f = 150$ and $p_d = 100$, Normal demand truncated at zero).

Buy-up makes protecting seats more valuable: turning a discount customer away no longer means a lost sale with certainty, so the airline should keep *fewer* discount seats open. The optimal protection level should therefore rise above the independent-case answer of $\approx 46$. A revenue manager who ignores buy-up and treats the classes as independent sets the discount limit **too high**.

## Setup

Imports and a fixed random seed so the simulated demand draws are reproducible.

In [1]:
import numpy as np
from scipy.optimize import minimize_scalar
np.random.seed(666)

## Optimal protection level with buy-up

The objective is the same as the independent case, with one change: unmet discount demand is recaptured into full-fare demand at rate $\alpha$. Concretely, `unmet_discounted_demand = (discount_demand - discounted_sales) * alpha` is added to full-fare demand before computing full-fare sales. We then search for the protection level that maximizes expected revenue.

In [2]:
# Fares and capacity
full_fare_price = 150
discount_price  = 100
plane_capacity  = 100

num_simulations = 200_000

# Demand distributions (Normal, truncated below at 0)
full_fare_mean     = 56
full_fare_std      = 23
discount_fare_mean = 88
discount_fare_std  = 44

alpha = 0.3  # recapture rate: share of turned-away discount demand that buys up to full fare

def simulate_demand(mean, std, num_simulations):
    """Draw non-negative demand from a Normal, clipping negative draws to 0."""
    demand = np.random.normal(mean, std, num_simulations)
    return np.where(demand < 0, 0, demand)

# Draw once, outside the objective, so every protection level is evaluated on the same realizations
full_fare_demand = simulate_demand(full_fare_mean, full_fare_std, num_simulations)
discount_demand  = simulate_demand(discount_fare_mean, discount_fare_std, num_simulations)

def expected_revenue(protection_level):
    discounted_sales = np.minimum(plane_capacity - protection_level, discount_demand)
    # Buy-up: a fraction alpha of unmet discount demand spills into full-fare demand
    unmet_discounted_demand = (discount_demand - discounted_sales) * alpha
    full_fare_sales = np.minimum(plane_capacity - discounted_sales, full_fare_demand + unmet_discounted_demand)
    revenue = full_fare_price * full_fare_sales + discount_price * discounted_sales
    return -np.mean(revenue)  # negate: maximize revenue via a minimizer

result = minimize_scalar(expected_revenue, bounds=(0, plane_capacity), method='bounded')

print(f"The optimal protection level to maximize expected revenue is: {round(result.x, 2)}")

The optimal protection level to maximize expected revenue is: 75.27
